In [1]:

import pandas as pd

url = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/Mall_Customers.csv"
mall = pd.read_csv(url)

print(mall.head())
print("Dataset shape:", mall.shape)


   CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)
0           1    Male   19                  15                      39
1           2    Male   21                  15                      81
2           3  Female   20                  16                       6
3           4  Female   23                  16                      77
4           5  Female   31                  17                      40
Dataset shape: (200, 5)


In [2]:

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score
import pandas as pd

# Select features for clustering
X = mall[['Annual Income (k$)', 'Spending Score (1-100)']]
X_scaled = StandardScaler().fit_transform(X)

# Compare clustering algorithms
results = []

models = {
    "K-Means": KMeans(n_clusters=5, random_state=42, n_init=10),
    "Agglomerative": AgglomerativeClustering(n_clusters=5),
    "DBSCAN": DBSCAN(eps=0.5, min_samples=5)
}

for name, model in models.items():
    labels = model.fit_predict(X_scaled)
    clusters = set(labels) - {-1}

    if len(clusters) >= 2:
        mask = labels != -1 if -1 in labels else [True] * len(labels)
        score = silhouette_score(X_scaled[mask], labels[mask])
    else:
        score = None

    results.append({
        "Algorithm": name,
        "Clusters Found": len(clusters),
        "Silhouette Score": round(score, 3) if score is not None else "Not applicable"
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


    Algorithm  Clusters Found  Silhouette Score
      K-Means               5             0.555
Agglomerative               5             0.554
       DBSCAN               2             0.388


In [5]:

import os

print(os.listdir('/content'))


['.config', 'sample_data']


In [8]:

import pandas as pd

insurance = pd.read_csv('/content/insurance_pre.csv')

print(insurance.head())
print("Columns:", insurance.columns.tolist())
print("Dataset shape:", insurance.shape)


   age     sex     bmi  children smoker      charges
0   19  female  27.900         0    yes  16884.92400
1   18    male  33.770         1     no   1725.55230
2   28    male  33.000         3     no   4449.46200
3   33    male  22.705         0     no  21984.47061
4   32    male  28.880         0     no   3866.85520
Columns: ['age', 'sex', 'bmi', 'children', 'smoker', 'charges']
Dataset shape: (1338, 6)


In [9]:

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

# Prepare data
X = insurance.drop('charges', axis=1)
y = insurance['charges']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), ['sex', 'smoker'])
], remainder='passthrough')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train and compare boosting models
models = {
    'AdaBoost': AdaBoostRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'Hist Gradient Boosting': HistGradientBoostingRegressor(random_state=42)
}

results = []

for name, model in models.items():
    pipe = make_pipeline(preprocessor, model)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    results.append({
        'Algorithm': name,
        'MAE': round(mean_absolute_error(y_test, pred), 2),
        'RMSE': round(np.sqrt(mean_squared_error(y_test, pred)), 2),
        'R2 Score': round(r2_score(y_test, pred), 4)
    })

boost_results = pd.DataFrame(results).sort_values('R2 Score', ascending=False)
print(boost_results.to_string(index=False))
print("\nBest boosting model:", boost_results.iloc[0]['Algorithm'])


             Algorithm     MAE    RMSE  R2 Score
     Gradient Boosting 2428.89 4344.26    0.8784
Hist Gradient Boosting 2577.51 4607.72    0.8632
              AdaBoost 4253.85 5181.86    0.8270

Best boosting model: Gradient Boosting
